# 07 · Re-run the 9 failed runs

Standalone: clones the fixed code, needs nothing attached.

* notebook 01, EBMT: `ours_nllpch_event_ranking` s0, `ours_nllpch_mmv_rank` s0 and s4
  (a validation step produced no C_td and checkpoint selection crashed);
* notebook 05, METABRIC: both LLMs, seeds 0–2 (single-event labels crashed the answer encoding).

Same tags, overrides and splits as the originals, so the results slot into those tables.
GPU session + Internet on.

## Options

In [ ]:
NAME = "07_rerun_failed"

MMV_LAMBDA_V = 1.0   # as in notebook 01
LLM_EPOCHS   = 30    # as in notebook 05
LLMS = {             # as in notebook 05
    "llm_pretrained": ["llm_init=pretrained", "llm_model=distilgpt2", "llm_learning_rate=5e-5"],
    "llm_scratch":    ["llm_init=scratch", "llm_learning_rate=5e-4"],
}

# --- execution --------------------------------------------------------------
SMOKE_TEST   = False    # False = the real run. True: 2 epochs, 1 seed, reduced plan, only to check
                        # that everything runs (~10-30 min); smoke results go to a separate folder.
SMOKE_EPOCHS = 2
WORKERS      = None     # parallel runs; None = 2 on Kaggle (4 vCPU), 4 locally
TIME_BUDGET_MIN = 11 * 60   # stop launching new runs after this (Kaggle sessions end at 12 h)
INSTALL_DEPS = True     # Kaggle only
GIT_URL      = "https://github.com/Parsagh05/dmmst.git"   # code is cloned from here (nothing to upload)
CLEAN_UP_AT_END = True  # Kaggle: delete the working repo copy (models, caches) after the report

## Setup (identical in every notebook)

In [ ]:
# ---- 1. get the code --------------------------------------------------------------
import os, sys, json, shutil, subprocess, time, hashlib
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").is_dir()

def _is_repo(p):
    return (p / "sat" / "finetune.py").is_file() and (p / "conf").is_dir()

if ON_KAGGLE:
    WORK = Path("/kaggle/working")
    REPO = WORK / "dmmst"
    if not _is_repo(REPO):
        # prefer the code dataset over repo copies inside attached notebook outputs (stale code)
        cands = [f.parent.parent for f in Path("/kaggle/input").rglob("finetune.py")
                 if f.parent.name == "sat" and _is_repo(f.parent.parent)]
        cands.sort(key=lambda p: ("code" not in str(p).lower(), len(p.parts)))
        src = cands[0] if cands else None
        if src is not None:
            print("copying code from attached dataset:", src)
            shutil.copytree(src, REPO, ignore=shutil.ignore_patterns(
                ".git", ".venv", "model-hub", "__pycache__", "results", "tb_logs"))
        elif GIT_URL:
            subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(REPO)], check=True)
        else:
            raise RuntimeError("Attach the dmmst code dataset (notebooks/version_1/README.md, step 1) or set GIT_URL.")
    RESULTS_ROOT = WORK / "results"
else:
    REPO = Path.cwd().resolve()
    while not _is_repo(REPO) and REPO != REPO.parent:
        REPO = REPO.parent
    RESULTS_ROOT = REPO / "results" / "new"
assert _is_repo(REPO), REPO
os.chdir(REPO)
sys.path.insert(0, str(REPO))
CODE_FINGERPRINT = hashlib.md5(b"".join(
    p.read_bytes() for p in sorted((REPO / "sat").rglob("*.py")) + sorted((REPO / "conf").rglob("*.yaml"))
)).hexdigest()[:10]
print("repo:", REPO, "| code fingerprint:", CODE_FINGERPRINT)

In [ ]:
# ---- 2. dependencies (Kaggle) ------------------------------------------------------
# numpy is NEVER moved: pandas/pyarrow in the image are compiled against it.
if ON_KAGGLE and INSTALL_DEPS:
    import numpy as _np, torch
    _tv = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
    pkgs = ["transformers==4.50.0", "datasets==3.4.1", "tokenizers>=0.21,<0.22", "evaluate>=0.4.3",
            "accelerate>=1.4.0", "hydra-core>=1.3.2", "hydra-colorlog>=1.2.0",
            "torchsurv" if _tv >= (2, 8) else "torchsurv==0.1.4",
            "h5py>=3.13", "logdecorator>=2.5", "einops==0.8.1", "torchtuples>=0.2.2",
            "numba", "nvidia-ml-py", "polars", "tqdm", "tabulate",
            "scikit-survival" if int(_np.__version__.split(".")[0]) >= 2 else "scikit-survival==0.22.2",
            f"numpy=={_np.__version__}"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
import torch
print("torch", torch.__version__, "| cuda" if torch.cuda.is_available() else "| cpu")

In [ ]:
# ---- 3. runner ----------------------------------------------------------------------
from IPython.display import Markdown, display
from scripts.runner import Run, Runner, dataset_overrides
from scripts.report import build as build_report

RUN_NAME = NAME + ("_smoke" if SMOKE_TEST else "")
r = Runner(repo=REPO, results_dir=RESULTS_ROOT / RUN_NAME,
           workers=WORKERS or (2 if ON_KAGGLE else 4),
           smoke_epochs=SMOKE_EPOCHS if SMOKE_TEST else None, gpu="0")

# Resume: attach this notebook's previous output ("Add Input" -> your notebook) and the
# finished runs are copied back, so only the missing ones are trained.
if ON_KAGGLE:
    n_copied = 0
    for d in Path("/kaggle/input").rglob(RUN_NAME):
        if d.is_dir() and d != r.results_dir:
            for f in d.glob("*.json"):
                if not (r.results_dir / f.name).exists():
                    shutil.copy2(f, r.results_dir / f.name); n_copied += 1
    print(f"resumed {n_copied} files from attached inputs")
(r.results_dir / "environment.json").write_text(json.dumps({
    "code_fingerprint": CODE_FINGERPRINT, "torch": torch.__version__,
    "cuda": torch.cuda.is_available(), "smoke": SMOKE_TEST, "name": RUN_NAME}, indent=2))
print("results ->", r.results_dir)

## Datasets

In [ ]:
# ---- datasets -------------------------------------------------------------------------
def ensure_dataset(ds):
    """Simulated scenarios and EBMT are generated on the fly; hsa_synthetic ships with the repo."""
    if ds == "hsa_synthetic" or (REPO / "data" / ds / "meta.json").is_file():
        return
    if ds == "ebmt":
        subprocess.run([sys.executable, "scripts/prepare_ebmt.py"], check=True)
    elif ds.startswith("sim_"):
        subprocess.run([sys.executable, "-m", "sat.data.simulate", "--scenario", ds[4:],
                        "--out", f"data/{ds}"], check=True, stdout=subprocess.DEVNULL)
    else:
        raise ValueError(ds)

def group(ds):
    return "hsa_synthetic" if ds == "hsa_synthetic" else "multievent"

def ds_overrides(ds):
    # hsa keeps its own configs; score every event at its own horizons like the others
    return ["per_event_horizons=true"] if ds == "hsa_synthetic" else dataset_overrides(ds, REPO)

def has_truth(ds):
    return (REPO / "data" / ds / "truth.npz").is_file()

ensure_dataset("ebmt")
r.prepare("multievent/survival", ds_overrides("ebmt"), label="ebmt")
r.prepare("survtrace_metabric/survival", [], label="metabric")

## The 9 failed runs (same tags and settings as the originals)

In [ ]:
runs = []
# notebook 01 - EBMT (checkpoint-selection crash, fixed in survtrace_metrics.py)
for rec, s in [("nllpch_event_ranking", 0), ("nllpch_mmv_rank", 0), ("nllpch_mmv_rank", 4)]:
    ov = ds_overrides("ebmt") + [f"seed={s}", f"split_seed={s}", f"tasks/losses={rec}"]
    info = {"dataset": "ebmt", "seed": s, "model": "ours", "recipe": rec, "source_notebook": "01"}
    if "mmv" in rec:
        ov.append(f"mmv_variance_weight={MMV_LAMBDA_V}")
        info["lambda_v"] = MMV_LAMBDA_V
    runs.append(Run(f"ebmt__ours_{rec}__s{s}", "finetune", "multievent/survival", ov, "ebmt", info))
# notebook 05 - METABRIC LLMs (single-event labels crashed the answer encoding, fixed in llm.py)
for s in [0, 1, 2]:
    for name, llm_ov in LLMS.items():
        runs.append(Run(f"metabric__{name}__s{s}", "llm", "survtrace_metabric/survival",
                        [f"seed={s}", f"split_seed={s}", f"llm_epochs={LLM_EPOCHS}"] + llm_ov,
                        "metabric_numeric",
                        {"dataset": "metabric", "seed": s, "model": name, "source_notebook": "05"}))
if SMOKE_TEST:
    runs = [runs[0], runs[3]]
print(len(runs), "runs")
r.run_many(runs, time_budget_min=TIME_BUDGET_MIN)

## Results
Copy the `.json` / `.meta.json` files whose `source_notebook` is 01 into notebook 01's results folder and those from 05 into notebook 05's, then re-run notebook 04; or attach this notebook's output to 04 directly.

In [ ]:
# ---- report + zip -----------------------------------------------------------------------
df = build_report([r.results_dir], r.results_dir / "report")
rep = r.results_dir / "report" / "REPORT.md"
if rep.is_file():
    display(Markdown(rep.read_text(encoding="utf-8")))
print("zip:", r.zip())

## Clean up
Keeps the Kaggle output small (results + zip only) and stops a later session from picking up this repo copy instead of the code dataset.

In [ ]:
if ON_KAGGLE and CLEAN_UP_AT_END:
    shutil.rmtree(REPO, ignore_errors=True)
    print("removed", REPO, "- results are in", r.results_dir)